In [1]:
pip install --pre bibtexparser

  Using cached bibtexparser-2.0.0b8-py3-none-any.whl.metadata (5.4 kB)
Using cached bibtexparser-2.0.0b8-py3-none-any.whl (39 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bibtexparser]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import bibtexparser
bibtex_str = "../bib_custom.bib"
library = bibtexparser.parse_string(bibtex_str) 

In [5]:
if len(library.failed_blocks) > 0:
    print("Some blocks failed to parse. Check the entries of `library.failed_blocks`.")
else:
    print("All blocks parsed successfully")

All blocks parsed successfully


In [13]:
len(library.entries_dict)

0

In [11]:
library.entries_dict

{}

In [16]:
from pybtex.database import parse_file
bibtex_str = "../bib_custom.bib"
bib_data = parse_file(bibtex_str)

In [19]:
len(bib_data.entries)

481

In [28]:
bib_data.entries['allcott2017social']

Entry('article',
  fields=[
    ('title', 'Social media and fake news in the 2016 election'),
    ('journal', 'Journal of economic perspectives'),
    ('volume', '31'),
    ('number', '2'),
    ('pages', '211--236'),
    ('year', '2017'),
    ('publisher', 'American Economic Association 2014 Broadway, Suite 305, Nashville, TN 37203-2418')],
  persons={'author': [Person('Allcott, Hunt'), Person('Gentzkow, Matthew')]})

In [1]:
import re
from pybtex.database import parse_file

In [2]:
def normalize_bibtex_text(s: str) -> str:
    s = str(s).replace("\n", " ")
    # fix missing space after 'and'
    s = re.sub(r'\band(?=\S)', 'and ', s)
    # normalize common LaTeX tokens
    s = s.replace("{-}", "-").replace("{\\%}", "%").replace("{%}", "%")
    # drop remaining braces
    s = s.replace("{", "").replace("}", "")
    # collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def clean_text(s: str, keep_digits=True) -> str:
    s = str(s).lower().strip().replace("\n", " ")
    s = re.sub(r'\s+', ' ', s)
    if keep_digits:
        s = re.sub(r'[^a-z0-9 ]', '', s)
    else:
        s = re.sub(r'[^a-z ]', '', s)
    return s.strip()

def get_authors(entry) -> str:
    persons = entry.persons.get('author', [])
    if persons:  # preferred path
        raw = " and ".join(str(p) for p in persons)
    else:        # fallback for malformed records
        raw = normalize_bibtex_text(entry.fields.get('author', ''))
        parts = re.split(r'\s+and\s+', raw)
        raw = " and ".join(p.strip() for p in parts if p.strip())
    return clean_text(raw, keep_digits=True)
    
def check_metadata_duplicates(bib_file_path):
    bib_data = parse_file(bib_file_path)
    entries = bib_data.entries
    seen = {}
    duplicates = []

    for key, entry in entries.items():
        # title = str(entry.fields.get('title', '')).lower().strip()
        # year = str(entry.fields.get('year', '')).strip()
        # author = " and ".join(str(p) for p in entry.persons.get('author', [])).lower().strip()
        
        title  = clean_text(normalize_bibtex_text(entry.fields.get('title', '')), keep_digits=True)
        year   = re.sub(r'[^0-9]', '', normalize_bibtex_text(entry.fields.get('year', '')))
        # author = get_authors(entry)

        # entry_id = (title, author, year)
        entry_id = (title, year)

        if entry_id in seen:
            duplicates.append((key, seen[entry_id]))
        else:
            seen[entry_id] = key

    if duplicates:
        print("Duplicate entries based on metadata (title, author, year):")
        for dup_key, original_key in duplicates:
            print(f" - {dup_key} is a duplicate of {original_key}")
    else:
        print("No duplicate metadata entries found.")

bibtex_str = "../bib_custom.bib"
# Example usage
check_metadata_duplicates(bibtex_str)


No duplicate metadata entries found.
